In [2]:
from pathlib import Path
import gc
import json
import random
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import (
    MinMaxScaler,
    Normalizer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Activation,
    BatchNormalization,
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    Input,
    MaxPooling1D,
)
from tensorflow.keras.optimizers import Adam

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


In [3]:
from os import name
column_names = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty",
]




original_test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Test.csv", names = column_names,header=None)
original_train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Train.csv", names = column_names,header=None)

In [4]:
original_train_df["binary_label"] = original_train_df["label"].where(
    original_train_df["label"].eq("normal"),
    "attack"
)
original_test_df["binary_label"] = original_test_df["label"].where(
    original_test_df["label"].eq("normal"),
    "attack"
)

In [5]:
original_test_df.drop(columns=["difficulty"], inplace=True)
original_train_df.drop(columns=["difficulty"], inplace=True)
original_train_df.loc[original_train_df["label"] == "normal", "label"] = "benign"
original_test_df.loc[original_test_df["label"] == "normal", "label"] = "benign"

In [6]:
darpa_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/Darpa/Darpa_Pcap_KDD_features.csv")
unsw_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/UNSW-NB15/UNSW_Pcap_NSL_features.csv")
cic2017_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2017/CIC_Pcap_NSL_features.csv")
cic2018_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2018/CIC18_Pcap_KDD_features.csv")

In [7]:
darpa_pcap.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty_level',
       'attack_family'],
      dtype='object')

In [8]:
unsw_pcap.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'attack_type', 'label'],
      dtype='object')

In [9]:
cic2017_pcap.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'category'],
      dtype='object')

In [10]:
cic2018_pcap.columns

Index(['Unnamed: 0', 'duration', 'protocol_type', 'service', 'flag',
       'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label'],
      dtype='object')

In [11]:
cic2017_pcap.rename(columns={"category": "label"}, inplace=True)
unsw_pcap.drop(columns=["label"], inplace=True)
unsw_pcap.rename(columns={"attack_type": "label"}, inplace=True)
darpa_pcap.drop(columns=["label", "difficulty_level"], inplace=True)
darpa_pcap.rename(columns={"attack_family": "label"}, inplace=True)


In [12]:
darpa_pcap['source_pcap'] = "darpa"
unsw_pcap['source_pcap'] = "unsw-nb15"
cic2017_pcap['source_pcap'] = "cic2017"
cic2018_pcap['source_pcap'] = "cic2018"

In [13]:
data_list = [darpa_pcap,unsw_pcap,cic2017_pcap,cic2018_pcap]
df = pd.concat(data_list)
df.shape

(2884849, 44)

In [14]:
df['label'].value_counts()

,count
label,
Normal,1329132
normal,661240
BENIGN,661240
DoS Hulk,144392
dos_goldeneye,27779
DoS,15533
Probe,9652
DoS GoldenEye,7563
dos_slowloris,7423


In [15]:
df['source_pcap'].value_counts()

,count
source_pcap,
darpa,1078656
cic2017,822995
cic2018,696452
unsw-nb15,286746


In [ ]:
import re

LABEL_COLUMN = "label"

df["original_label"] = df[LABEL_COLUMN].copy()

cleaned_labels = (
    df[LABEL_COLUMN]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s*-\s*attempted\s*$",
        "",
        regex=True,
        flags=re.IGNORECASE,
    )
    .str.strip()
)

normalized_labels = cleaned_labels.str.casefold()

df[LABEL_COLUMN] = cleaned_labels

benign_mask = normalized_labels.isin(["normal", "benign"])
df.loc[benign_mask, LABEL_COLUMN] = "benign"

recon_mask = normalized_labels.isin(["probe", "reconnaissance"])
df.loc[recon_mask, LABEL_COLUMN] = "reconnaissance"

dos_mask = cleaned_labels.str.match(
    r"^(dos\b|dos[_\s-])",
    case=False,
    na=False,
)
df.loc[dos_mask, LABEL_COLUMN] = "dos"

display(df[LABEL_COLUMN].value_counts(dropna=False).to_frame("count"))

,count
label,
benign,2651612
dos,210758
reconnaissance,10513
Exploits,5371
Fuzzers,3760
Web Attack - Brute Force,1111
Generic,675
Web Attack - XSS,587
Shellcode,146


In [ ]:
source_label_counts = (
    df.groupby(["source_pcap", "label"], dropna=False)
      .size()
      .reset_index(name="count")
)

source_totals = (
    source_label_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

source_label_counts["within_source_pct"] = (
    100 * source_label_counts["count"] / source_totals
)

display(
    source_label_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,label,count,within_source_pct
5,cic2017,benign,661240,80.345567
6,cic2017,dos,160023,19.443982
2,cic2017,Web Attack - Brute Force,1111,0.134995
4,cic2017,Web Attack - XSS,587,0.071325
1,cic2017,Infiltration,21,0.002552
3,cic2017,Web Attack - Sql Injection,12,0.001458
0,cic2017,Heartbleed,1,0.000122
7,cic2018,benign,661240,94.944088
8,cic2018,dos,35202,5.054476
9,cic2018,infiltration,10,0.001436


In [18]:
protocol_counts = (
    df.groupby(["source_pcap", "protocol_type"], dropna=False)
      .size()
      .reset_index(name="count")
)

protocol_totals = (
    protocol_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

protocol_counts["within_source_pct"] = (
    100 * protocol_counts["count"] / protocol_totals
)

display(
    protocol_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,protocol_type,count,within_source_pct
1,cic2017,tcp,424414,51.569451
2,cic2017,udp,398279,48.393854
0,cic2017,icmp,302,0.036695
4,cic2018,tcp,679248,97.529765
5,cic2018,udp,9181,1.318253
3,cic2018,icmp,8023,1.151982
7,darpa,tcp,833984,77.316957
8,darpa,udp,240635,22.308781
6,darpa,icmp,4037,0.374262
10,unsw-nb15,tcp,212939,74.260495


In [19]:
source_label_protocol = (
    df.groupby(
        ["source_pcap", "label", "protocol_type"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

display(
    source_label_protocol.sort_values(
        ["source_pcap", "label", "count"],
        ascending=[True, True, False]
    )
)

,source_pcap,label,protocol_type,count
0,cic2017,Heartbleed,tcp,1
1,cic2017,Infiltration,tcp,21
2,cic2017,Web Attack - Brute Force,tcp,1111
3,cic2017,Web Attack - Sql Injection,tcp,12
4,cic2017,Web Attack - XSS,tcp,587
7,cic2017,benign,udp,398279
6,cic2017,benign,tcp,262660
5,cic2017,benign,icmp,301
9,cic2017,dos,tcp,160022
8,cic2017,dos,icmp,1


In [ ]:
source_share = (
    df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("overall_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

attack_df = df[df["label"] != "benign"]

attack_source_share = (
    attack_df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("attack_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

display(source_share)
display(attack_source_share)

,source_pcap,overall_share_pct
0,darpa,37.390380
1,cic2017,28.528183
2,cic2018,24.141714
3,unsw-nb15,9.939723


,source_pcap,attack_share_pct
0,cic2017,69.352204
1,cic2018,15.097090
2,darpa,10.564790
3,unsw-nb15,4.985916


In [ ]:
df_master = df.copy()

df_master["binary_label"] = df_master["label"].where(
    df_master["label"].eq("benign"),
    "attack"
)

matched_category_sources = {
    "dos": {"darpa", "unsw-nb15", "cic2017", "cic2018"},
    "reconnaissance": {"darpa", "unsw-nb15"},
}

def assign_category_scope(row):
    label = row["label"]
    source = row["source_pcap"]

    if label == "benign":
        return "benign"

    valid_sources = matched_category_sources.get(label)

    if valid_sources is not None and source in valid_sources:
        return "matched_category"

    return "source_specific_or_unmatched"

df_master["category_scope"] = df_master.apply(
    assign_category_scope,
    axis=1
)

df_master["feature_schema"] = "kdd_nsl"

scope_summary = (
    df_master
    .groupby(
        ["source_pcap", "label", "binary_label", "category_scope"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["source_pcap", "category_scope", "count"],
        ascending=[True, True, False]
    )
)

display(scope_summary)

,source_pcap,label,binary_label,category_scope,count
5,cic2017,benign,benign,benign,661240
6,cic2017,dos,attack,matched_category,160023
2,cic2017,Web Attack - Brute Force,attack,source_specific_or_unmatched,1111
4,cic2017,Web Attack - XSS,attack,source_specific_or_unmatched,587
1,cic2017,Infiltration,attack,source_specific_or_unmatched,21
3,cic2017,Web Attack - Sql Injection,attack,source_specific_or_unmatched,12
0,cic2017,Heartbleed,attack,source_specific_or_unmatched,1
7,cic2018,benign,benign,benign,661240
8,cic2018,dos,attack,matched_category,35202
9,cic2018,infiltration,attack,source_specific_or_unmatched,10


In [ ]:
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
PREPARATION_SEED = 42
EXTERNAL_TEST_SIZE = 0.30
VALIDATION_SIZE = 0.20

SOURCE_COLUMN = "source_pcap"
LABEL_COLUMN = "label"
BINARY_COLUMN = "binary_label"
GROUP_COLUMN = None

ORIGINAL_TRAIN_VARIABLE = "original_train_df"
ORIGINAL_TEST_VARIABLE = "original_test_df"

FEATURE_COLUMNS = column_names[:41]
CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]

MODEL_KEYS = [
    "kwon_cnn",
    "vinayakumar_dnn",
    "kilichev_cnn",
    "rf_selected_knn",
    "ckan",
]
SEEDS = [11, 22, 33, 44, 55]

MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

EPOCH_OVERRIDES = {
    "kwon_cnn": 10,
    "vinayakumar_dnn": 20,
    "kilichev_cnn": 72,
    "ckan": 20,
}
BATCH_SIZE_OVERRIDES = {
    "kwon_cnn": 32,
    "vinayakumar_dnn": 64,
    "kilichev_cnn": 32,
    "rf_selected_knn": 1024,
    "ckan": 32,
}

TARGET_TEST_NORMAL_FRACTION = 0.431
TARGET_TEST_ATTACK_FRACTION = 0.569
EXP4_EXPERIMENT = "exp4_mixed_traffic_test_ratio43p1_normal"

OUTPUT_DIR = Path("nsl_kdd_exp4_ratio43p1_normal_prepared")
RESULTS_DIR = Path("nsl_kdd_exp4_ratio43p1_normal_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAVE_RATIO_MATCHED_TEST_SETS = False

print("Target: 43.1% normal / 56.9% attack")
print("Results:", RESULTS_DIR.resolve())

Target: 43.1% normal / 56.9% attack
Results: /content/nsl_kdd_exp4_ratio43p1_normal_results


In [ ]:

FEATURE_COLUMNS = column_names[:41]


def normalize_label(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().rstrip(".").casefold()


def make_binary_target(frame):
    labels = frame[LABEL_COLUMN].map(normalize_label)
    return np.where(labels.eq("normal") | labels.eq("benign"), 0, 1).astype(np.int32)


def validate_frame(frame, name):
    missing = sorted(set(FEATURE_COLUMNS + [LABEL_COLUMN]) - set(frame.columns))
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")


def downsample_frame(frame, max_rows, seed):
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)

    target = make_binary_target(frame)
    sampled, _ = train_test_split(
        frame,
        train_size=max_rows,
        random_state=seed,
        stratify=target,
    )
    return sampled.reset_index(drop=True)


def load_csv(name):
    path = DATA_DIR.resolve() / name
    print(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required prepared dataset not found: {path}"
        )
    frame = pd.read_csv(path, low_memory=False)
    validate_frame(frame, name)
    return frame


def source_name(frame, fallback):
    if SOURCE_COLUMN in frame.columns and frame[SOURCE_COLUMN].nunique() == 1:
        return str(frame[SOURCE_COLUMN].iloc[0])
    return fallback



In [ ]:

def get_dataframe(variable_name):
    if variable_name not in globals():
        raise NameError(
            f"{variable_name!r} is not defined. Load the original KDD/NSL "
            "DataFrame or change the corresponding configuration variable."
        )
    frame = globals()[variable_name]
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{variable_name!r} must be a pandas DataFrame.")
    return frame.copy()


def validate_columns(frame, required, frame_name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f"{frame_name} is missing columns: {missing}")


def ensure_binary_label(frame):
    result = frame.copy()
    result[BINARY_COLUMN] = np.where(
        result[LABEL_COLUMN].eq("benign"),
        "benign",
        "attack",
    )
    return result


def source_label_distribution(frame, label_column=LABEL_COLUMN):
    table = (
        frame.groupby([SOURCE_COLUMN, label_column], dropna=False)
        .size()
        .reset_index(name="count")
    )
    table["within_source_pct"] = (
        100 * table["count"]
        / table.groupby(SOURCE_COLUMN)["count"].transform("sum")
    )
    return table.sort_values(
        [SOURCE_COLUMN, "count"],
        ascending=[True, False],
    )


def split_each_source(frame, stratify_column):
    """Split every source separately and preserve source-specific test sets."""
    train_parts = []
    test_parts = []

    for source, source_frame in frame.groupby(SOURCE_COLUMN, sort=True):
        source_frame = source_frame.copy()

        if GROUP_COLUMN is not None and GROUP_COLUMN in source_frame.columns:
            splitter = GroupShuffleSplit(
                n_splits=1,
                test_size=EXTERNAL_TEST_SIZE,
                random_state=RANDOM_STATE,
            )
            train_idx, test_idx = next(
                splitter.split(
                    source_frame,
                    groups=source_frame[GROUP_COLUMN],
                )
            )
            train_part = source_frame.iloc[train_idx].copy()
            test_part = source_frame.iloc[test_idx].copy()
        else:
            if GROUP_COLUMN is not None:
                warnings.warn(
                    f"{source}: group column {GROUP_COLUMN!r} was not found; "
                    "using record-level splitting."
                )

            counts = source_frame[stratify_column].value_counts(dropna=False)
            stratify_values = (
                source_frame[stratify_column]
                if counts.min() >= 2
                else None
            )
            if stratify_values is None:
                warnings.warn(
                    f"{source}: at least one {stratify_column} class contains "
                    "fewer than two records; using an unstratified split."
                )

            train_part, test_part = train_test_split(
                source_frame,
                test_size=EXTERNAL_TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=stratify_values,
            )

        train_part["split"] = "train"
        test_part["split"] = "test"
        train_parts.append(train_part)
        test_parts.append(test_part)

    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
    )


def balance_shared_external_training(frame):

    selected_attack_parts = []
    allocation = []

    for category in ["dos", "reconnaissance"]:
        category_frame = frame[frame[LABEL_COLUMN].eq(category)]
        counts = category_frame.groupby(SOURCE_COLUMN).size()
        present_sources = sorted(counts.index.tolist())

        if len(present_sources) < 2:
            warnings.warn(
                f"Skipping {category}: it is present in fewer than two sources."
            )
            continue

        quota = int(counts.min())

        for source in present_sources:
            cell = category_frame[category_frame[SOURCE_COLUMN].eq(source)]
            selected = cell.sample(
                n=quota,
                replace=False,
                random_state=RANDOM_STATE,
            )
            selected_attack_parts.append(selected)
            allocation.append({
                "label": category,
                "source_pcap": source,
                "available": len(cell),
                "selected": quota,
            })

    if not selected_attack_parts:
        raise ValueError("No shared attack data were available for balancing.")

    selected_attacks = pd.concat(selected_attack_parts, ignore_index=True)
    selected_benign_parts = []

    for source, attack_count in selected_attacks.groupby(SOURCE_COLUMN).size().items():
        benign_cell = frame[
            frame[SOURCE_COLUMN].eq(source)
            & frame[LABEL_COLUMN].eq("benign")
        ]
        if len(benign_cell) < attack_count:
            raise ValueError(
                f"{source} has {len(benign_cell)} benign rows but requires "
                f"{attack_count} for 1:1 benign/attack balancing."
            )
        selected_benign_parts.append(
            benign_cell.sample(
                n=int(attack_count),
                replace=False,
                random_state=RANDOM_STATE,
            )
        )

    balanced = pd.concat(
        [selected_attacks, *selected_benign_parts],
        ignore_index=True,
    )
    balanced = balanced.sample(
        frac=1,
        random_state=RANDOM_STATE,
    ).reset_index(drop=True)

    return balanced, pd.DataFrame(allocation)


def balance_binary_external_training(frame):
    """
    Strict source × binary-class balancing for Experiment 4.

    Every source contributes the same number of benign and attack records,
    determined by the smallest source/class training cell.
    """
    counts = (
        frame.groupby([SOURCE_COLUMN, BINARY_COLUMN])
        .size()
        .unstack(fill_value=0)
    )

    if not {"benign", "attack"}.issubset(counts.columns):
        raise ValueError(f"Both binary classes are required per source:\n{counts}")

    quota = int(counts[["benign", "attack"]].min().min())
    if quota < 1:
        raise ValueError("A source/class cell contains zero records.")

    parts = []
    for source in sorted(frame[SOURCE_COLUMN].unique()):
        for binary_class in ["benign", "attack"]:
            cell = frame[
                frame[SOURCE_COLUMN].eq(source)
                & frame[BINARY_COLUMN].eq(binary_class)
            ]
            parts.append(
                cell.sample(
                    n=quota,
                    replace=False,
                    random_state=RANDOM_STATE,
                )
            )

    balanced = pd.concat(parts, ignore_index=True)
    balanced = balanced.sample(
        frac=1,
        random_state=RANDOM_STATE,
    ).reset_index(drop=True)
    return balanced, quota


def save_frame(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")
    return path



def _largest_remainder_allocation(counts, requested_total):
    """Allocate a sample across strata while preserving proportions."""
    counts = pd.Series(counts, dtype="int64")
    available_total = int(counts.sum())

    if requested_total < 0 or requested_total > available_total:
        raise ValueError(
            f"requested_total={requested_total} exceeds "
            f"available_total={available_total}"
        )

    if requested_total == 0:
        return counts * 0

    raw = counts / available_total * requested_total
    allocation = np.floor(raw).astype("int64")
    remaining = int(requested_total - allocation.sum())
    fractional = (raw - allocation).sort_values(ascending=False)

    while remaining > 0:
        progressed = False
        for stratum in fractional.index:
            if allocation.loc[stratum] < counts.loc[stratum]:
                allocation.loc[stratum] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError("Could not complete proportional allocation.")

    return allocation


def proportional_stratified_sample(
    frame,
    requested_total,
    strata_column,
    random_state,
):

    if requested_total == len(frame):
        return frame.copy()

    strata = (
        frame[strata_column]
        .astype("string")
        .fillna("<missing>")
    )
    counts = strata.value_counts(sort=False)
    allocation = _largest_remainder_allocation(
        counts,
        requested_total,
    )

    sampled_parts = []
    ordered_strata = sorted(allocation.index.astype(str))

    for offset, stratum in enumerate(ordered_strata):
        take = int(allocation.loc[stratum])
        if take == 0:
            continue

        cell = frame.loc[strata.eq(stratum)]
        sampled_parts.append(
            cell.sample(
                n=take,
                replace=False,
                random_state=random_state + offset,
            )
        )

    if not sampled_parts:
        return frame.iloc[0:0].copy()

    return pd.concat(sampled_parts, axis=0)


def match_binary_test_ratio(
    frame,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
):

    if not 0.0 < target_attack_fraction < 1.0:
        raise ValueError("target_attack_fraction must be between 0 and 1.")

    working = frame.copy()
    working["__ratio_original_order"] = np.arange(len(working))
    working["__ratio_binary"] = make_binary_target(working)

    attack_rows = working[working["__ratio_binary"].eq(1)]
    normal_rows = working[working["__ratio_binary"].eq(0)]

    available_attack = int(len(attack_rows))
    available_normal = int(len(normal_rows))

    if available_attack == 0 or available_normal == 0:
        raise ValueError(
            "Every ratio-matched test source must contain both attack "
            "and normal rows."
        )

    target_normal_fraction = 1.0 - target_attack_fraction
    maximum_total = min(
        int(np.floor(available_attack / target_attack_fraction)),
        int(np.floor(available_normal / target_normal_fraction)),
    )

    while maximum_total > 1:
        selected_attack = int(
            round(maximum_total * target_attack_fraction)
        )
        selected_normal = maximum_total - selected_attack

        if (
            0 < selected_attack <= available_attack
            and 0 < selected_normal <= available_normal
        ):
            break

        maximum_total -= 1
    else:
        raise ValueError("Not enough rows to construct a matched test set.")

    sampled_attack = proportional_stratified_sample(
        attack_rows,
        selected_attack,
        LABEL_COLUMN,
        random_state,
    )
    sampled_normal = normal_rows.sample(
        n=selected_normal,
        replace=False,
        random_state=random_state + 10_000,
    )

    matched = (
        pd.concat([sampled_attack, sampled_normal], axis=0)
        .sort_values("__ratio_original_order")
        .drop(
            columns=[
                "__ratio_original_order",
                "__ratio_binary",
            ]
        )
        .reset_index(drop=True)
    )

    actual_attack_fraction = float(make_binary_target(matched).mean())

    audit = {
        "available_rows": int(len(frame)),
        "available_attack": available_attack,
        "available_normal": available_normal,
        "selected_rows": int(len(matched)),
        "selected_attack": int(selected_attack),
        "selected_normal": int(selected_normal),
        "target_attack_pct": 100.0 * target_attack_fraction,
        "target_normal_pct": 100.0 * target_normal_fraction,
        "actual_attack_pct": 100.0 * actual_attack_fraction,
        "actual_normal_pct": 100.0 * (1.0 - actual_attack_fraction),
    }

    if round(audit["actual_normal_pct"], 1) != round(
        audit["target_normal_pct"], 1
    ):
        raise AssertionError(
            "The matched test set does not round to the requested "
            f"normal prevalence: {audit}"
        )

    return matched, audit


def ratio_match_test_sets(
    test_sets,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
):
    matched_sets = {}
    audit_rows = []

    for offset, (test_source, test_frame) in enumerate(
        test_sets.items()
    ):
        matched, audit = match_binary_test_ratio(
            test_frame,
            target_attack_fraction=target_attack_fraction,
            random_state=random_state + 1_000 * offset,
        )
        matched["ratio_match"] = "43.1% normal / 56.9% attack"
        matched_sets[test_source] = matched
        audit_rows.append({
            "test_source": test_source,
            **audit,
        })

    return matched_sets, pd.DataFrame(audit_rows)



In [ ]:

original_train = ensure_binary_label(
    get_dataframe(ORIGINAL_TRAIN_VARIABLE)
)
original_test = ensure_binary_label(
    get_dataframe(ORIGINAL_TEST_VARIABLE)
)
external_master = ensure_binary_label(df_master)

validate_columns(
    external_master,
    set(FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN]),
    "external_master",
)
validate_columns(
    original_train,
    set(FEATURE_COLUMNS + [LABEL_COLUMN]),
    "original_train",
)
validate_columns(
    original_test,
    set(FEATURE_COLUMNS + [LABEL_COLUMN]),
    "original_test",
)

exp4_external_train_natural, exp4_external_test_natural = split_each_source(
    external_master,
    stratify_column=BINARY_COLUMN,
)

exp4_external_train_balanced, exp4_source_class_quota = (
    balance_binary_external_training(exp4_external_train_natural)
)

exp4_train_natural = pd.concat(
    [original_train, exp4_external_train_natural],
    ignore_index=True,
    sort=False,
)
exp4_train_balanced = pd.concat(
    [original_train, exp4_external_train_balanced],
    ignore_index=True,
    sort=False,
)

exp4_natural_tests = {
    "original": original_test.reset_index(drop=True),
    **{
        source: part.reset_index(drop=True)
        for source, part in exp4_external_test_natural.groupby(
            SOURCE_COLUMN,
            sort=True,
        )
    },
}

exp4_test_sets, exp4_test_ratio_manifest = ratio_match_test_sets(
    exp4_natural_tests,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
)

print(
    "Balanced external quota:",
    f"{exp4_source_class_quota:,} rows per source/class",
)
print("Mixed balanced train:", exp4_train_balanced.shape)
display(exp4_test_ratio_manifest)

for source, frame in exp4_test_sets.items():
    target = make_binary_target(frame)
    print(
        source,
        frame.shape,
        f"normal={(target == 0).mean() * 100:.3f}%",
        f"attack={(target == 1).mean() * 100:.3f}%",
    )

Balanced external quota: 8,140 rows per source/class
Mixed balanced train: (191093, 49)


,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,22544,12833,9711,22531,12820,9711,56.9,43.1,56.899383,43.100617
1,cic2017,246899,48527,198372,85284,48527,36757,56.9,43.1,56.900474,43.099526
2,cic2018,208936,10564,198372,18565,10563,8002,56.9,43.1,56.897388,43.102612
3,darpa,323597,7392,316205,12991,7392,5599,56.9,43.1,56.900931,43.099069
4,unsw-nb15,86024,3489,82535,6131,3489,2642,56.9,43.1,56.907519,43.092481


original (22531, 44) normal=43.101% attack=56.899%
cic2017 (85284, 50) normal=43.100% attack=56.900%
cic2018 (18565, 50) normal=43.103% attack=56.897%
darpa (12991, 50) normal=43.099% attack=56.901%
unsw-nb15 (6131, 50) normal=43.092% attack=56.908%


In [ ]:
exp4_split_manifest = pd.DataFrame([
    ["official_nsl_kdd_train", len(original_train)],
    ["external_train_natural", len(exp4_external_train_natural)],
    ["external_train_balanced", len(exp4_external_train_balanced)],
    ["combined_train_natural", len(exp4_train_natural)],
    ["combined_train_balanced", len(exp4_train_balanced)],
    ["external_test_natural", len(exp4_external_test_natural)],
    [
        "ratio_matched_test_rows",
        sum(len(frame) for frame in exp4_test_sets.values()),
    ],
], columns=["dataset", "rows"])

exp4_split_manifest.to_csv(
    RESULTS_DIR / f"{EXP4_EXPERIMENT}_split_manifest.csv",
    index=False,
)
exp4_test_ratio_manifest.to_csv(
    RESULTS_DIR / f"{EXP4_EXPERIMENT}_test_ratio_manifest.csv",
    index=False,
)

if SAVE_RATIO_MATCHED_TEST_SETS:
    for source, frame in exp4_test_sets.items():
        frame.to_csv(
            RESULTS_DIR / f"{EXP4_EXPERIMENT}_test_{source}.csv",
            index=False,
        )

display(exp4_split_manifest)
display(exp4_test_ratio_manifest)

if GROUP_COLUMN is None:
    print(
        "External splitting is record-level. Document this limitation "
        "unless capture/session grouping metadata is available."
    )

,dataset,rows
0,official_nsl_kdd_train,125973
1,external_train_natural,2019393
2,external_train_balanced,65120
3,combined_train_natural,2145366
4,combined_train_balanced,191093
5,external_test_natural,865456
6,ratio_matched_test_rows,145502


,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,22544,12833,9711,22531,12820,9711,56.9,43.1,56.899383,43.100617
1,cic2017,246899,48527,198372,85284,48527,36757,56.9,43.1,56.900474,43.099526
2,cic2018,208936,10564,198372,18565,10563,8002,56.9,43.1,56.897388,43.102612
3,darpa,323597,7392,316205,12991,7392,5599,56.9,43.1,56.900931,43.099069
4,unsw-nb15,86024,3489,82535,6131,3489,2642,56.9,43.1,56.907519,43.092481


External splitting is record-level. Document this limitation unless capture/session grouping metadata is available.


In [ ]:

def calculate_binary_metrics(y_true, y_pred, y_probability=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    result = {
        "n_test": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": (tn / (tn + fp) if (tn + fp) else np.nan),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) else np.nan
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if y_probability is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, y_probability)
    else:
        result["roc_auc"] = np.nan

    return result


def category_recall_table(frame, y_true, y_pred):
    labels = frame[LABEL_COLUMN].astype(str).to_numpy()

    rows = []
    for category in sorted(pd.unique(labels)):
        mask = labels == category
        category_true = y_true[mask]
        category_pred = y_pred[mask]

        rows.append({
            "test_category": category,
            "support": int(mask.sum()),
            "binary_attack_rate": float(category_true.mean()),
            "binary_recall": recall_score(
                category_true,
                category_pred,
                pos_label=1,
                zero_division=0,
            ) if np.any(category_true == 1) else np.nan,
            "benign_specificity": recall_score(
                category_true,
                category_pred,
                pos_label=0,
                zero_division=0,
            ) if np.any(category_true == 0) else np.nan,
        })

    return pd.DataFrame(rows)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:

def _prepare_test_frame(test_frame, seed):
    return downsample_frame(test_frame, MAX_TEST_ROWS, seed)


def _keras_binary_prediction(model, X, batch_size):
    probabilities = model.predict(
        X,
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)
    predictions = (probabilities >= 0.5).astype(np.int32)
    return predictions, probabilities


In [29]:

def fit_kwon(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_train_df = train_frame[FEATURE_COLUMNS].copy()
    y_train = make_binary_target(train_frame)

    numerical_columns = [
        column for column in FEATURE_COLUMNS
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_columns),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    print("    Fitting preprocessing...", flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_train = np.expand_dims(X_train, axis=-1)

    model = Sequential([
        Input(shape=(X_train.shape[1], 1)),
        Conv1D(64, 3, strides=1, padding="same", use_bias=False),
        Activation("relu"),
        BatchNormalization(),
        Dropout(0.5),
        MaxPooling1D(pool_size=2, strides=2),
        Flatten(),
        Dense(64, use_bias=False),
        Activation("relu"),
        BatchNormalization(),
        Dropout(0.5),
        Dense(2, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print("    Training Kwon CNN...", flush=True)
    model.fit(
        X_train,
        y_train,
        epochs=EPOCH_OVERRIDES["kwon_cnn"],
        batch_size=BATCH_SIZE_OVERRIDES["kwon_cnn"],
        shuffle=True,
        verbose=1,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
        "n_train": len(train_frame),
    }


def predict_kwon(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted["seed"])
    X_test = fitted["preprocessor"].transform(
        test_frame[FEATURE_COLUMNS]
    ).astype(np.float32)
    X_test = np.expand_dims(X_test, axis=-1)

    probabilities = fitted["model"].predict(
        X_test,
        batch_size=BATCH_SIZE_OVERRIDES["kwon_cnn"],
        verbose=0,
    )
    predictions = np.argmax(probabilities, axis=1).astype(np.int32)

    return (
        make_binary_target(test_frame),
        predictions,
        probabilities[:, 1],
        test_frame,
    )


In [ ]:


def build_vinayakumar_model(input_dim=None):
    if input_dim is None:
        input_dim = len(FEATURE_COLUMNS)

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(
                shape=(input_dim,),
                name="nsl_kdd_input",
            ),

            tf.keras.layers.Dense(
                units=1024,
                activation="relu",
                name="dense_1024",
            ),
            tf.keras.layers.BatchNormalization(
                name="batch_norm_1024",
            ),
            tf.keras.layers.Dropout(
                rate=0.01,
                name="dropout_1024",
            ),

            tf.keras.layers.Dense(
                units=768,
                activation="relu",
                name="dense_768",
            ),
            tf.keras.layers.BatchNormalization(
                name="batch_norm_768",
            ),
            tf.keras.layers.Dropout(
                rate=0.01,
                name="dropout_768",
            ),

            tf.keras.layers.Dense(
                units=512,
                activation="relu",
                name="dense_512",
            ),
            tf.keras.layers.BatchNormalization(
                name="batch_norm_512",
            ),
            tf.keras.layers.Dropout(
                rate=0.01,
                name="dropout_512",
            ),

            tf.keras.layers.Dense(
                units=256,
                activation="relu",
                name="dense_256",
            ),
            tf.keras.layers.BatchNormalization(
                name="batch_norm_256",
            ),
            tf.keras.layers.Dropout(
                rate=0.01,
                name="dropout_256",
            ),

            tf.keras.layers.Dense(
                units=128,
                activation="relu",
                name="dense_128",
            ),
            tf.keras.layers.BatchNormalization(
                name="batch_norm_128",
            ),
            tf.keras.layers.Dropout(
                rate=0.01,
                name="dropout_128",
            ),

            tf.keras.layers.Dense(
                units=1,
                activation="sigmoid",
                name="binary_output",
            ),
        ],
        name="nsl_kdd_binary_dnn",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.1,
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy",
                threshold=0.5,
            ),
            tf.keras.metrics.Precision(
                name="precision",
                thresholds=0.5,
            ),
            tf.keras.metrics.Recall(
                name="recall",
                thresholds=0.5,
            ),
            tf.keras.metrics.AUC(
                name="auc",
            ),
        ],
    )

    return model




def fit_vinayakumar(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame).astype(np.float32)

    X_train_df, X_val_df, y_train, y_val = train_test_split(
        X_all_df,
        y_all,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=y_all,
    )

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    X_train_df = X_train_df.copy()
    X_val_df = X_val_df.copy()

    X_train_df.loc[:, CATEGORICAL_COLUMNS] = encoder.fit_transform(
        X_train_df[CATEGORICAL_COLUMNS]
    )
    X_val_df.loc[:, CATEGORICAL_COLUMNS] = encoder.transform(
        X_val_df[CATEGORICAL_COLUMNS]
    )

    X_train = X_train_df.astype(np.float32).to_numpy()
    X_val = X_val_df.astype(np.float32).to_numpy()

    normalizer = Normalizer(norm="l2")
    X_train = normalizer.fit_transform(X_train).astype(np.float32)
    X_val = normalizer.transform(X_val).astype(np.float32)

    model = build_vinayakumar_model()

    print("    Training Vinayakumar DNN...", flush=True)
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCH_OVERRIDES["vinayakumar_dnn"],
        batch_size=BATCH_SIZE_OVERRIDES["vinayakumar_dnn"],
        shuffle=True,
        verbose=1,
    )

    return {
        "model": model,
        "encoder": encoder,
        "normalizer": normalizer,
        "seed": seed,
        "n_train": len(train_frame),
    }


def predict_vinayakumar(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted["seed"])
    X_test_df = test_frame[FEATURE_COLUMNS].copy()

    X_test_df.loc[:, CATEGORICAL_COLUMNS] = fitted["encoder"].transform(
        X_test_df[CATEGORICAL_COLUMNS]
    )

    X_test = X_test_df.astype(np.float32).to_numpy()
    X_test = fitted["normalizer"].transform(X_test).astype(np.float32)

    predictions, probabilities = _keras_binary_prediction(
        fitted["model"],
        X_test,
        BATCH_SIZE_OVERRIDES["vinayakumar_dnn"],
    )

    return make_binary_target(test_frame), predictions, probabilities, test_frame


In [ ]:

def fit_kilichev(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    y_all = make_binary_target(train_frame)

    retained_features = [
        feature for feature in FEATURE_COLUMNS
        if feature != "num_outbound_cmds"
    ]

    X_raw = train_frame[retained_features].copy()

    numerical_columns = [
        column for column in retained_features
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", "passthrough", numerical_columns),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    print("    Fitting encoding and chi-square selection...", flush=True)
    X_all = preprocessor.fit_transform(X_raw).astype(np.float32)

    selector = SelectKBest(
        score_func=chi2,
        k=min(100, X_all.shape[1]),
    )
    X_all = selector.fit_transform(X_all, y_all).astype(np.float32)

    X_train, X_val, y_train, y_val = train_test_split(
        X_all,
        y_all,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=y_all,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)

    X_train = np.expand_dims(X_train, axis=-1)
    X_val = np.expand_dims(X_val, axis=-1)

    model = Sequential([
        Input(shape=(X_train.shape[1], 1)),
        Conv1D(32, 9, activation="relu"),
        MaxPooling1D(pool_size=5),
        Conv1D(64, 9, activation="relu"),
        MaxPooling1D(pool_size=5),
        Dropout(0.253378),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.253378),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(learning_rate=0.000471),
        metrics=["accuracy"],
    )

    print("    Training Kilichev CNN...", flush=True)
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE_OVERRIDES["kilichev_cnn"],
        epochs=EPOCH_OVERRIDES["kilichev_cnn"],
        verbose=1,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selector": selector,
        "scaler": scaler,
        "retained_features": retained_features,
        "seed": seed,
        "n_train": len(train_frame),
    }


def predict_kilichev(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted["seed"])

    X_test = fitted["preprocessor"].transform(
        test_frame[fitted["retained_features"]]
    ).astype(np.float32)
    X_test = fitted["selector"].transform(X_test).astype(np.float32)
    X_test = fitted["scaler"].transform(X_test).astype(np.float32)
    X_test = np.expand_dims(X_test, axis=-1)

    predictions, probabilities = _keras_binary_prediction(
        fitted["model"],
        X_test,
        BATCH_SIZE_OVERRIDES["kilichev_cnn"],
    )

    return make_binary_target(test_frame), predictions, probabilities, test_frame


In [ ]:

def fit_rf_selected_knn(train_frame, seed):
    set_all_seeds(seed)

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)

    X_train_df, _, y_train, _ = train_test_split(
        X_all_df,
        y_all,
        test_size=0.30,
        random_state=seed,
        stratify=y_all,
        shuffle=True,
    )

    numerical_columns = [
        column for column in FEATURE_COLUMNS
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                MinMaxScaler(feature_range=(0, 1), clip=True),
                numerical_columns,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    print("    Fitting preprocessing and Random Forest selector...", flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)

    random_forest = RandomForestClassifier(
        n_estimators=300,
        random_state=seed,
        n_jobs=-1,
    )
    random_forest.fit(X_train, y_train)

    selected_mask = random_forest.feature_importances_ >= 0.037
    if not np.any(selected_mask):
        selected_mask[np.argmax(random_forest.feature_importances_)] = True

    knn = KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        metric="euclidean",
        n_jobs=-1,
    )

    print("    Training KNN...", flush=True)
    knn.fit(X_train[:, selected_mask], y_train)

    return {
        "model": knn,
        "preprocessor": preprocessor,
        "selected_mask": selected_mask,
        "seed": seed,
        "n_train": len(train_frame),
    }


def predict_rf_selected_knn(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted["seed"])
    X_test = fitted["preprocessor"].transform(
        test_frame[FEATURE_COLUMNS]
    ).astype(np.float32)
    X_test = X_test[:, fitted["selected_mask"]]

    probabilities = fitted["model"].predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int32)

    return make_binary_target(test_frame), predictions, probabilities, test_frame


In [ ]:
class KANLinear(nn.Module):

    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        grid_range=(-1.0, 1.0),
    ):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.number_of_spline_coefficients = (
            grid_size + spline_order
        )

        grid_step = (
            grid_range[1] - grid_range[0]
        ) / grid_size

        grid = (
            torch.arange(
                -spline_order,
                grid_size + spline_order + 1,
                dtype=torch.float32,
            )
            * grid_step
            + grid_range[0]
        )

        grid = grid.expand(
            in_features,
            -1,
        ).contiguous()

        self.register_buffer(
            "grid",
            grid,
        )

        self.base_weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )

        self.spline_weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
                self.number_of_spline_coefficients,
            )
        )

        self.spline_scaler = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )

        nn.init.kaiming_uniform_(
            self.base_weight,
            a=np.sqrt(5),
        )

        nn.init.normal_(
            self.spline_weight,
            mean=0.0,
            std=0.02,
        )

        nn.init.ones_(
            self.spline_scaler
        )


    def b_splines(self, x):



        x_expanded = x.unsqueeze(-1)

        grid = self.grid

        bases = (
            (
                x_expanded >= grid[:, :-1]
            )
            &
            (
                x_expanded < grid[:, 1:]
            )
        ).to(x.dtype)

        for order in range(
            1,
            self.spline_order + 1,
        ):
            left_numerator = (
                x_expanded
                - grid[:, :-(order + 1)]
            )

            left_denominator = (
                grid[:, order:-1]
                - grid[:, :-(order + 1)]
            )

            right_numerator = (
                grid[:, order + 1:]
                - x_expanded
            )

            right_denominator = (
                grid[:, order + 1:]
                - grid[:, 1:-order]
            )

            left_term = (
                left_numerator
                / left_denominator
                * bases[:, :, :-1]
            )

            right_term = (
                right_numerator
                / right_denominator
                * bases[:, :, 1:]
            )

            bases = left_term + right_term

        return bases.contiguous()


    def forward(self, x):

        base_output = F.linear(
            F.silu(x),
            self.base_weight,
        )

        spline_bases = self.b_splines(x)

        scaled_spline_weights = (
            self.spline_weight
            * self.spline_scaler.unsqueeze(-1)
        )

        spline_output = F.linear(
            spline_bases.reshape(
                x.shape[0],
                -1,
            ),
            scaled_spline_weights.reshape(
                self.out_features,
                -1,
            ),
        )

        return base_output + spline_output

In [ ]:

def fit_ckan(train_frame, seed):
    set_all_seeds(seed)

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)

    X_train_df, X_val_df, y_train, y_val = train_test_split(
        X_all_df,
        y_all,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=y_all,
    )

    numerical_columns = [
        column for column in FEATURE_COLUMNS
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                MinMaxScaler(feature_range=(0, 1), clip=True),
                numerical_columns,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    print("    Fitting CKAN preprocessing...", flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_val = preprocessor.transform(X_val_df).astype(np.float32)

    train_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(X_train).unsqueeze(1),
            torch.from_numpy(y_train.astype(np.int64)),
        ),
        batch_size=BATCH_SIZE_OVERRIDES["ckan"],
        shuffle=True,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CKANBinaryDynamic().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=0.01,
    )

    print("    Training CKAN...", flush=True)
    for epoch in range(EPOCH_OVERRIDES["ckan"]):
        model.train()
        running_loss = 0.0

        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch_labels)

        print(
            f"      epoch {epoch + 1}/{EPOCH_OVERRIDES['ckan']} "
            f"loss={running_loss / len(train_loader.dataset):.6f}",
            flush=True,
        )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "device": device,
        "seed": seed,
        "n_train": len(train_frame),
    }


def predict_ckan(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted["seed"])
    X_test = fitted["preprocessor"].transform(
        test_frame[FEATURE_COLUMNS]
    ).astype(np.float32)

    test_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(X_test).unsqueeze(1),
            torch.from_numpy(
                make_binary_target(test_frame).astype(np.int64)
            ),
        ),
        batch_size=BATCH_SIZE_OVERRIDES["ckan"],
        shuffle=False,
    )

    fitted["model"].eval()
    predictions = []
    probabilities = []

    with torch.no_grad():
        for batch_features, _ in test_loader:
            batch_features = batch_features.to(fitted["device"])
            logits = fitted["model"](batch_features)
            probs = torch.softmax(logits, dim=1)

            predictions.extend(torch.argmax(probs, dim=1).cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())

    return (
        make_binary_target(test_frame),
        np.asarray(predictions, dtype=np.int32),
        np.asarray(probabilities, dtype=np.float32),
        test_frame,
    )


In [35]:
class CKANBinaryDynamic(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=0,
            bias=True,
        )
        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=0,
            bias=True,
        )
        self.adaptive_pool = nn.AdaptiveAvgPool1d(output_size=1)
        self.kan = KANLinear(
            in_features=64,
            out_features=2,
            grid_size=5,
            spline_order=3,
            grid_range=(-1.0, 1.0),
        )

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.adaptive_pool(x).squeeze(-1)
        return self.kan(x)


def fit_predict_ckan(train_frame, test_frame, seed):
    set_all_seeds(seed)

    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    test_frame = downsample_frame(test_frame, MAX_TEST_ROWS, seed)

    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    X_test_df = test_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)
    y_test = make_binary_target(test_frame)

    X_train_df, X_val_df, y_train, y_val = train_test_split(
        X_all_df,
        y_all,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=y_all,
    )

    numerical_columns = [
        column for column in FEATURE_COLUMNS
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                MinMaxScaler(
                    feature_range=(0.0, 1.0),
                    clip=True,
                ),
                numerical_columns,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_val = preprocessor.transform(X_val_df).astype(np.float32)
    X_test = preprocessor.transform(X_test_df).astype(np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(X_train).unsqueeze(1),
        torch.from_numpy(y_train.astype(np.int64)),
    )
    validation_dataset = TensorDataset(
        torch.from_numpy(X_val).unsqueeze(1),
        torch.from_numpy(y_val.astype(np.int64)),
    )
    test_dataset = TensorDataset(
        torch.from_numpy(X_test).unsqueeze(1),
        torch.from_numpy(y_test.astype(np.int64)),
    )

    batch_size = BATCH_SIZE_OVERRIDES["ckan"]
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = CKANBinaryDynamic().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=0.01,
    )

    for _ in range(EPOCH_OVERRIDES["ckan"]):
        model.train()
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()

    model.eval()
    predictions = []
    probabilities = []

    with torch.no_grad():
        for batch_features, _ in test_loader:
            batch_features = batch_features.to(device)
            logits = model(batch_features)
            probs = torch.softmax(logits, dim=1)

            predictions.extend(
                torch.argmax(probs, dim=1).cpu().numpy()
            )
            probabilities.extend(probs[:, 1].cpu().numpy())

    return (
        y_test,
        np.asarray(predictions, dtype=np.int32),
        np.asarray(probabilities, dtype=np.float32),
    )


In [ ]:

MODEL_REGISTRY = {
    "kwon_cnn": {
        "fit": fit_kwon,
        "predict": predict_kwon,
    },
    "vinayakumar_dnn": {
        "fit": fit_vinayakumar,
        "predict": predict_vinayakumar,
    },
    "kilichev_cnn": {
        "fit": fit_kilichev,
        "predict": predict_kilichev,
    },
    "rf_selected_knn": {
        "fit": fit_rf_selected_knn,
        "predict": predict_rf_selected_knn,
    },
    "ckan": {
        "fit": fit_ckan,
        "predict": predict_ckan,
    },
}


unknown_models = sorted(set(MODEL_KEYS) - set(MODEL_REGISTRY))
if unknown_models:
    raise KeyError(f"Unknown MODEL_KEYS: {unknown_models}")

print("Selected models:", MODEL_KEYS)


Selected models: ['kwon_cnn', 'vinayakumar_dnn', 'kilichev_cnn', 'rf_selected_knn', 'ckan']


In [37]:
def _merge_and_save(path, new_frame, duplicate_columns):
    if path.exists():
        previous = pd.read_csv(path)
        combined = pd.concat(
            [previous, new_frame],
            ignore_index=True,
        )
    else:
        combined = new_frame.copy()

    combined = combined.drop_duplicates(
        subset=duplicate_columns,
        keep="last",
    )
    combined.to_csv(path, index=False)
    return combined


def evaluate_model_on_test_sets(
    model_key,
    train_frame,
    test_sets,
    experiment,
    training_variant,
    seed,
):
    adapter = MODEL_REGISTRY[model_key]

    print(
        f"\n{experiment} | {model_key} | "
        f"{training_variant} | seed={seed}",
        flush=True,
    )
    print("  Training once...", flush=True)

    fit_started = time.perf_counter()
    fitted = adapter["fit"](train_frame, seed)
    fit_seconds = time.perf_counter() - fit_started

    print(
        f"  Training completed in "
        f"{fit_seconds / 60:.2f} minutes.",
        flush=True,
    )

    overall_rows = []
    category_parts = []

    pooled_true = []
    pooled_prediction = []
    pooled_score = []

    for test_source, test_frame in test_sets.items():
        print(
            f"  Testing {test_source} "
            f"({len(test_frame):,} raw rows)...",
            flush=True,
        )

        prediction_started = time.perf_counter()
        (
            y_true,
            y_pred,
            y_score,
            evaluated_test,
        ) = adapter["predict"](fitted, test_frame)
        prediction_seconds = (
            time.perf_counter() - prediction_started
        )

        metrics = calculate_binary_metrics(
            y_true,
            y_pred,
            y_score,
        )

        evaluated_attack_fraction = float(np.mean(y_true == 1))

        overall_rows.append({
            "experiment": experiment,
            "training_variant": training_variant,
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            "n_train_raw": int(len(train_frame)),
            "fit_seconds": fit_seconds,
            "prediction_seconds": prediction_seconds,
            "test_attack_pct": 100.0 * evaluated_attack_fraction,
            "test_normal_pct": 100.0 * (1.0 - evaluated_attack_fraction),
            **metrics,
        })

        categories = category_recall_table(
            evaluated_test,
            y_true,
            y_pred,
        )
        categories["experiment"] = experiment
        categories["training_variant"] = training_variant
        categories["model"] = model_key
        categories["seed"] = seed
        categories["test_source"] = test_source
        category_parts.append(categories)

        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)

        print(
            f"    evaluated={len(y_true):,}, "
            f"accuracy={metrics['accuracy']:.4f}, "
            f"recall={metrics['recall']:.4f}, "
            f"f1={metrics['f1']:.4f}",
            flush=True,
        )

    pooled_true = np.concatenate(pooled_true)
    pooled_prediction = np.concatenate(pooled_prediction)
    pooled_score = np.concatenate(pooled_score)

    pooled_metrics = calculate_binary_metrics(
        pooled_true,
        pooled_prediction,
        pooled_score,
    )

    pooled_attack_fraction = float(np.mean(pooled_true == 1))

    overall_rows.append({
        "experiment": experiment,
        "training_variant": training_variant,
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "n_train_raw": int(len(train_frame)),
        "fit_seconds": fit_seconds,
        "prediction_seconds": sum(
            row["prediction_seconds"]
            for row in overall_rows
        ),
        "test_attack_pct": 100.0 * pooled_attack_fraction,
        "test_normal_pct": 100.0 * (1.0 - pooled_attack_fraction),
        **pooled_metrics,
    })

    del fitted
    gc.collect()
    tf.keras.backend.clear_session()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        pd.DataFrame(overall_rows),
        pd.concat(category_parts, ignore_index=True),
    )


def run_experiment(
    experiment,
    train_variants,
    test_sets,
):
    overall_path = (
        RESULTS_DIR
        / f"{experiment}_all_runs_partial.csv"
    )
    category_path = (
        RESULTS_DIR
        / f"{experiment}_category_runs_partial.csv"
    )

    completed = (
        pd.read_csv(overall_path)
        if overall_path.exists()
        else pd.DataFrame()
    )

    overall_accumulated = completed.copy()
    category_accumulated = (
        pd.read_csv(category_path)
        if category_path.exists()
        else pd.DataFrame()
    )

    expected_sources = set(test_sets) | {
        "POOLED_ALL_ROWS"
    }

    for model_key in MODEL_KEYS:
        for training_variant, train_frame in train_variants.items():
            for seed in SEEDS:
                if not completed.empty:
                    mask = (
                        completed["experiment"].eq(experiment)
                        & completed["model"].eq(model_key)
                        & completed[
                            "training_variant"
                        ].eq(training_variant)
                        & completed["seed"].eq(seed)
                    )
                    completed_sources = set(
                        completed.loc[
                            mask,
                            "test_source",
                        ].astype(str)
                    )
                else:
                    completed_sources = set()

                if expected_sources.issubset(
                    completed_sources
                ):
                    print(
                        f"SKIP completed: {model_key}, "
                        f"{training_variant}, seed={seed}",
                        flush=True,
                    )
                    continue

                overall, categories = (
                    evaluate_model_on_test_sets(
                        model_key=model_key,
                        train_frame=train_frame,
                        test_sets=test_sets,
                        experiment=experiment,
                        training_variant=training_variant,
                        seed=seed,
                    )
                )

                overall_accumulated = _merge_and_save(
                    overall_path,
                    overall,
                    [
                        "experiment",
                        "training_variant",
                        "model",
                        "seed",
                        "test_source",
                    ],
                )

                category_accumulated = _merge_and_save(
                    category_path,
                    categories,
                    [
                        "experiment",
                        "training_variant",
                        "model",
                        "seed",
                        "test_source",
                        "test_category",
                    ],
                )

                completed = overall_accumulated

    return overall_accumulated, category_accumulated


In [ ]:

def summarize_overall(results):
    metrics = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "macro_f1",
        "mcc",
        "false_positive_rate",
        "roc_auc",
    ]

    return (
        results
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "test_source",
            ],
            dropna=False,
        )[metrics]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


def summarize_categories(results):
    return (
        results
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "test_source",
                "test_category",
            ],
            dropna=False,
        )
        .agg(
            support=("support", "first"),
            binary_recall_mean=(
                "binary_recall", "mean"
            ),
            binary_recall_std=(
                "binary_recall", "std"
            ),
            benign_specificity_mean=(
                "benign_specificity", "mean"
            ),
            benign_specificity_std=(
                "benign_specificity", "std"
            ),
        )
        .reset_index()
    )


def source_macro_summary(results):
    source_only = results[
        results["test_source"].ne(
            "POOLED_ALL_ROWS"
        )
    ]

    by_seed = (
        source_only
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "seed",
            ]
        )
        .agg(
            source_macro_accuracy=(
                "accuracy", "mean"
            ),
            source_macro_balanced_accuracy=(
                "balanced_accuracy", "mean"
            ),
            source_macro_f1=("f1", "mean"),
            source_macro_mcc=("mcc", "mean"),
            source_macro_recall=("recall", "mean"),
            source_macro_specificity=("specificity", "mean"),
        )
        .reset_index()
    )

    return (
        by_seed
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
            ]
        )
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


def save_final_results(
    experiment,
    overall,
    categories,
):
    overall_summary = summarize_overall(overall)
    category_summary = summarize_categories(categories)
    source_summary = source_macro_summary(overall)

    overall.to_csv(
        RESULTS_DIR / f"{experiment}_all_runs.csv",
        index=False,
    )
    categories.to_csv(
        RESULTS_DIR / f"{experiment}_category_runs.csv",
        index=False,
    )
    overall_summary.to_csv(
        RESULTS_DIR / f"{experiment}_summary.csv",
        index=False,
    )
    category_summary.to_csv(
        RESULTS_DIR / f"{experiment}_category_summary.csv",
        index=False,
    )
    source_summary.to_csv(
        RESULTS_DIR / f"{experiment}_source_macro_summary.csv",
        index=False,
    )

    return (
        overall_summary,
        category_summary,
        source_summary,
    )


In [ ]:
exp4_overall, exp4_categories = run_experiment(
    experiment=EXP4_EXPERIMENT,
    train_variants={
        "mixed_external_balanced": exp4_train_balanced,

    },
    test_sets=exp4_test_sets,
)

(
    exp4_summary,
    exp4_category_summary,
    exp4_source_macro_summary,
) = save_final_results(
    EXP4_EXPERIMENT,
    exp4_overall,
    exp4_categories,
)

exp4_test_ratio_manifest.to_csv(
    RESULTS_DIR / f"{EXP4_EXPERIMENT}_test_ratio_manifest.csv",
    index=False,
)

print("Row-weighted result: test_source == 'POOLED_ALL_ROWS'")
print("Equal-source result: source_macro_summary CSV")

display(exp4_test_ratio_manifest)
display(exp4_summary)
display(exp4_category_summary)
display(exp4_source_macro_summary)


exp4_mixed_traffic_test_ratio43p1_normal | kwon_cnn | mixed_external_balanced | seed=11
  Training once...
    Fitting preprocessing...
    Training Kwon CNN...
Epoch 1/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9458 - loss: 0.1439
Epoch 2/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.9598 - loss: 0.1093
Epoch 3/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9627 - loss: 0.1006
Epoch 4/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9651 - loss: 0.0948
Epoch 5/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9668 - loss: 0.0925
Epoch 6/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9672 - loss: 0.0914
Epoch 7/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9685 - loss: 0.0879
Epoch 8/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9690 - loss: 0.0869
Epoch 9/10
5972/5972 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9698 - loss: 0.0850
Epoch 10/10
5972/5972 ━━━━━━━━━━━━━━━

,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,22544,12833,9711,22531,12820,9711,56.9,43.1,56.899383,43.100617
1,cic2017,246899,48527,198372,85284,48527,36757,56.9,43.1,56.900474,43.099526
2,cic2018,208936,10564,198372,18565,10563,8002,56.9,43.1,56.897388,43.102612
3,darpa,323597,7392,316205,12991,7392,5599,56.9,43.1,56.900931,43.099069
4,unsw-nb15,86024,3489,82535,6131,3489,2642,56.9,43.1,56.907519,43.092481


experiment         training_variant  \
                                                                        
0   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
1   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
2   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
3   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
4   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
5   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
6   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
7   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
8   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
9   exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
10  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
11  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
12  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
13  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
14  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
15  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
16  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
17  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
18  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
19  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
20  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
21  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
22  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
23  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
24  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
25  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
26  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
27  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
28  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
29  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   

              model      test_source  accuracy                                \
                                          mean       std       min       max   
0              ckan  POOLED_ALL_ROWS  0.920512  0.004544  0.915458  0.926681   
1              ckan          cic2017  0.963090  0.006737  0.953919  0.972164   
2              ckan          cic2018  0.931915  0.007303  0.919903  0.938594   
3              ckan            darpa  0.886614  0.024886  0.842198  0.900316   
4              ckan         original  0.784217  0.012496  0.768097  0.802627   
5              ckan        unsw-nb15  0.866417  0.004338  0.859566  0.870005   
6      kilichev_cnn  POOLED_ALL_ROWS  0.951302  0.000977  0.950660  0.952977   
7      kilichev_cnn          cic2017  0.982892  0.001909  0.980477  0.984991   
8      kilichev_cnn          cic2018  0.990531  0.001697  0.988581  0.992513   
9      kilichev_cnn            darpa  0.976599  0.002023  0.973289  0.978524   
10     kilichev_cnn         original  0.796156  0.007829  0.785851  0.806311   
11     kilichev_cnn        unsw-nb15  0.909640  0.008697  0.898385  0.916979   
12         kwon_cnn  POOLED_ALL_ROWS  0.949689  0.002816  0.945925  0.952585   
13         kwon_cnn          cic2017  0.988584  0.000909  0.987419  0.989623   
14         kwon_cnn          cic2018  0.995723  0.001300  0.993698  0.997253   
15         kwon_cnn            darpa  0.968809  0.006742  0.957278  0.974290   
16         kwon_cnn         original  0.766251  0.014757  0.747326  0.784963   
17         kwon_cnn        unsw-nb15  0.902854  0.009844  0.891372  0.918284   
18  rf_selected_knn  POOLED_ALL_ROWS  0.922111  0.007927  0.907946  0.926077   
19  rf_selected_k

,experiment,training_variant,model,test_source,test_category,support,binary_recall_mean,binary_recall_std,benign_specificity_mean,benign_specificity_std
0,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,ckan,cic2017,Infiltration,4,0.650000,0.223607,NaN,NaN
1,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,ckan,cic2017,Web Attack - Brute Force,304,0.009868,0.011155,NaN,NaN
2,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,ckan,cic2017,Web Attack - Sql Injection,3,0.000000,0.000000,NaN,NaN
3,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,ckan,cic2017,Web Attack - XSS,180,0.001111,0.002485,NaN,NaN
4,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,ckan,cic2017,benign,36757,NaN,NaN,0.954273,0.025680
...,...,...,...,...,...,...,...,...,...,...
310,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,Shellcode,45,0.995556,0.009938,NaN,NaN
311,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,Worms,9,0.822222,0.099381,NaN,NaN
312,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,benign,2642,NaN,NaN,0.695685,0.315925
313,exp4_mixed_traffic_test_ratio43p1_normal,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,dos,205,0.887805,0.064068,NaN,NaN


experiment         training_variant  \
                                                                       
0  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
1  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
2  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
3  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   
4  exp4_mixed_traffic_test_ratio43p1_normal  mixed_external_balanced   

             model  seed                    source_macro_accuracy            \
                    mean        std min max                  mean       std   
0             ckan  33.0  17.392527  11  55              0.886451  0.005613   
1     kilichev_cnn  33.0  17.392527  11  55              0.931164  0.002142   
2         kwon_cnn  33.0  17.392527  11  55              0.924444  0.005264   
3  rf_selected_knn  33.0  17.392527  11  55              0.900042  0.004487   
4  vinayakumar_dnn  33.0  17.392527  11  55              0.878814  0.042225   

             ... source_macro_mcc           source_macro_recall            \
        min  ...              min       max                mean       std   
0  0.877006  ...         0.773576  0.789764            0.860074  0.025801   
1  0.927975  ...         0.863836  0.872518            0.911321  0.005849   
2  0.916376  ...         0.845661  0.867269            0.890675  0.010623   
3  0.892068  ...         0.797224  0.816457            0.874340  0.004896   
4  0.803779  ...         0.619844  0.808703            0.893830  0.043129   

                      source_macro_specificity                                
        min       max                     mean       std       min       max  
0  0.814326  0.876186                 0.921273  0.022195  0.903953  0.959757  
1  0.903143  0.918448                 0.957361  0.003318  0.951846  0.960761  
2  0.873851  0.902358                 0.969028  0.002875  0.965626  0.972521  
3  0.865711  0.877158                 0.933973  0.004140  0.926864  0.936679  
4  0.853340  0.964736                 0.858982  0.150382  0.591259  0.940558  

[5 rows x 31 columns]